# Live risk testing: pull all trades via Quantifi DSP
Read-only: lists every trade with HQL, then builds pricers and computes PV.

In [1]:
import pandas as pd
import os
import time

# Toolkit import so the pricer objects returned by DSP can be used
from Quantifi.Toolkit.Base import Dt

# Quantifi DSP REST client
from qservices.QuantifiApiClient import *

In [2]:
# Environment: switch between Prod and UAT here
ENV = 'Prod'   # 'Prod' or 'UAT'
host = {'Prod': 'hb-dc1pd-qfidsp', 'UAT': 'hb-dc1ut-qfidsp'}[ENV]
os.environ["Q_DATA_SERVICE_URL_PREFIX"] = f"https://{host}:47200"
os.environ["Q_MARKET_ENV_SERVICE_URL_PREFIX"] = f"https://{host}:47201"
os.environ["Q_LOCAL_INSTALL"] = "True"

requests.packages.urllib3.disable_warnings()   # unverified HTTPS cert

AS_OF = '2026-09-29'      # pricing date
CALC_ENV = 'Official'
MAX_TRADES = 100000       # data_query silently truncates at this many rows (default is 500)

## 1. List all trades

In [3]:
all_trades = Connect.data_query("FROM Trade b", max_results=MAX_TRADES, verify_https_cert=False)
print('Trades returned:', len(all_trades))
if len(all_trades) >= MAX_TRADES:
    print('WARNING: hit MAX_TRADES, results are probably truncated; raise it.')

trade_ids = [t['TradeId'] for t in all_trades]
print('Keys on a trade record:', sorted(all_trades[0].keys()) if all_trades else None)

Trades returned: 11313
Keys on a trade record: ['$type', 'Amount', 'BookingEntity', 'Counterparty', 'Groups', 'InitialMargin', 'IsCleared', 'Notes', 'ObjectId', 'Payment', 'PaymentCcy', 'PaymentSettle', 'Product', 'Settle', 'Strategy', 'SubStrategy', 'Tags', 'Termination', 'TradeId', 'TradeStatus', 'TradeType', 'Traded', 'TradedLevel', 'TradedLevelType', 'Trader']


In [ ]:
# Flat static view of the trades (top-level scalar fields only)
trades_static = pd.json_normalize(all_trades, max_level=1)


def flatten_tags(df, col, prefix):
    """Turn a column of [{'Name':..., 'Value':...}, ...] into one column per tag name."""
    def to_dict(tags):
        if not isinstance(tags, list):
            return {}
        return {f"{prefix}{t['Name']}": t.get('Value') for t in tags if isinstance(t, dict) and 'Name' in t}
    wide = pd.DataFrame([to_dict(t) for t in df[col]], index=df.index)
    return pd.concat([df.drop(columns=col), wide], axis=1)


for col, prefix in [('Tags', 'Tag.'), ('Product.Tags', 'Product.Tag.')]:
    if col in trades_static:
        trades_static = flatten_tags(trades_static, col, prefix)

display(trades_static.head(20))
trades_static.to_csv('all_trades_static.csv', index=False)

## 2. Build pricers and compute PV
Pricing is batched and errors are captured per chunk, so one bad trade does not kill the run.

In [5]:
from concurrent.futures import ThreadPoolExecutor, as_completed

CHUNK = 10        # matches the client's internal batch_size, so one HTTP request per task
WORKERS = 4       # lower to 1 if the server or pythonnet objects misbehave under concurrency
pricers, errors = {}, {}


def load_chunk(chunk):
    """Return (pricers, errors) for one chunk of trade ids."""
    try:
        # with_errors=True returns (pricers, errors) so per-trade server-side failures are captured
        p, e = Connect.get_pricers_by_trade_ids(as_of=AS_OF, trade_ids=chunk, calc_env=CALC_ENV,
                                                batch_size=len(chunk), with_errors=True,
                                                verify_https_cert=False)
        e = {k: '; '.join(map(str, v)) if isinstance(v, (list, tuple)) else str(v)
             for k, v in e.items()}
        return p, e
    except Exception:
        # whole request failed (e.g. HTTP error): fall back to one-by-one to isolate the failing trade(s)
        p, e = {}, {}
        for tid in chunk:
            try:
                pr, er = Connect.get_pricer_by_trade_id(as_of=AS_OF, trade_id=tid, calc_env=CALC_ENV,
                                                        with_errors=True, verify_https_cert=False)
                if pr is not None:
                    p[tid] = pr
                if er:
                    e[tid] = str(er)
            except Exception as ex:
                e[tid] = str(ex)
        return p, e


chunks = [trade_ids[i:i+CHUNK] for i in range(0, len(trade_ids), CHUNK)]
t0 = time.time()
done = 0
with ThreadPoolExecutor(max_workers=WORKERS) as pool:
    futures = [pool.submit(load_chunk, c) for c in chunks]
    for f in as_completed(futures):
        p, e = f.result()
        pricers.update(p)
        errors.update(e)
        done += 1
        if done % 10 == 0 or done == len(chunks):
            elapsed = time.time() - t0
            eta = elapsed / done * (len(chunks) - done)
            print(f'{done}/{len(chunks)} chunks | {len(pricers)} pricers, {len(errors)} errors | '
                  f'{elapsed:.0f}s elapsed, ~{eta:.0f}s left', flush=True)

print(f'Loaded {len(pricers)} pricers, {len(errors)} errors in {time.time()-t0:.0f}s')
if errors:
    display(pd.Series(errors, name='LoadError').head(20))

10/1132 chunks | 0 pricers, 100 errors | 9s elapsed, ~995s left
20/1132 chunks | 0 pricers, 200 errors | 17s elapsed, ~970s left
30/1132 chunks | 13 pricers, 240 errors | 27s elapsed, ~975s left
40/1132 chunks | 15 pricers, 320 errors | 36s elapsed, ~992s left
50/1132 chunks | 29 pricers, 380 errors | 40s elapsed, ~875s left
60/1132 chunks | 45 pricers, 410 errors | 43s elapsed, ~773s left
70/1132 chunks | 49 pricers, 480 errors | 48s elapsed, ~735s left
80/1132 chunks | 53 pricers, 560 errors | 54s elapsed, ~704s left
90/1132 chunks | 53 pricers, 660 errors | 61s elapsed, ~706s left
100/1132 chunks | 53 pricers, 760 errors | 68s elapsed, ~703s left
110/1132 chunks | 53 pricers, 860 errors | 76s elapsed, ~706s left
120/1132 chunks | 53 pricers, 960 errors | 83s elapsed, ~701s left
130/1132 chunks | 53 pricers, 1060 errors | 90s elapsed, ~694s left
140/1132 chunks | 53 pricers, 1160 errors | 98s elapsed, ~691s left
150/1132 chunks | 53 pricers, 1260 errors | 103s elapsed, ~677s left
160

FixToFloatBond_0257511      HTTP Response code: 400, with message [No Trad...
FixToFloatBond_0258672      HTTP Response code: 400, with message [No Trad...
FixToFloatBond_0258676      HTTP Response code: 400, with message [No Trad...
FixToFloatBond_0259734      HTTP Response code: 400, with message [No Trad...
FixToFloatBond_0261285      HTTP Response code: 400, with message [No Trad...
FixToFloatBond_0261829      HTTP Response code: 400, with message [No Trad...
FixToFloatBond_0263123      HTTP Response code: 400, with message [No Trad...
FixToFloatBond_0262730      HTTP Response code: 400, with message [No Trad...
FixToFloatBond_0262775      HTTP Response code: 400, with message [No Trad...
FixToFloatBond_0262891      HTTP Response code: 400, with message [No Trad...
Risk-FixToFltBond0228537    HTTP Response code: 400, with message [No Trad...
Risk-FixToFltBond0229728    HTTP Response code: 400, with message [No Trad...
FixToFloatBond_0229728      HTTP Response code: 400, with messag

In [6]:
rows = []
for tid, p in pricers.items():
    row = {'TradeId': tid, 'Pricer': type(p).__name__}
    try:
        row['Product'] = str(p.Product)
    except Exception:
        pass
    try:
        row['PV'] = p.Pv()
    except Exception as e:
        row['PvError'] = str(e)
    rows.append(row)

risk_df = pd.DataFrame(rows).set_index('TradeId')
display(risk_df.style.format({'PV': '{:,.2f}'}))
risk_df.to_csv('all_trades_pv.csv')

if errors:
    display(pd.Series(errors, name='LoadError'))

,Pricer,Product,PV
TradeId,,,
Bond_0245803,BondPricer,US-TSY 3.75% May30,0.00
Bond_0245903,BondPricer,US-TSY 3.75% May30,0.00
Bond0020145,BondPricer,OBL 2.2% Apr28,0.00
Bond0020160,BondPricer,OBL 2.2% Apr28,0.00
Bond_0245867,BondPricer,US-TSY 3.75% May30,0.00
Bond_0246526,BondPricer,OBL 2.2% Apr28,0.00
Bond_0243361,BondPricer,OBL 2.2% Apr28,0.00
Bond_0244424,BondPricer,OBL 2.2% Apr28,0.00
Bond0020161,BondPricer,OBL 2.2% Apr28,0.00


FixToFloatBond_0257511    HTTP Response code: 400, with message [No Trad...
FixToFloatBond_0258672    HTTP Response code: 400, with message [No Trad...
FixToFloatBond_0258676    HTTP Response code: 400, with message [No Trad...
FixToFloatBond_0259734    HTTP Response code: 400, with message [No Trad...
FixToFloatBond_0261285    HTTP Response code: 400, with message [No Trad...
                                                ...                        
BondFuture_0717298        HTTP Response code: 400, with message [No Trad...
BondFuture_0741003        HTTP Response code: 400, with message [No Trad...
BondFuture_0752174        HTTP Response code: 400, with message [No Trad...
BondFuture_0754471        HTTP Response code: 400, with message [No Trad...
BondFuture_0754472        HTTP Response code: 400, with message [No Trad...
Name: LoadError, Length: 9224, dtype: str